# Methode D - Hybride LLM-Transformation (Metadata + Vision)

**Concept:** The LLM receives both the parsed Figma JSON (structural metadata) and
the screenshot (visual evidence) to generate Vue 3 single-file components (SFCs).

**Three Context Strategies (Variants):**

| Variant | Context                                     | Character                                       |
|---------|---------------------------------------------|-------------------------------------------------|
| **D1**  | No documentation                            | minimal, no api reference, error-prone          |
| **D2**  | Docs only for detected components (raw)     | focused, relevant, but noisy                    |
| **D3**  | Docs only for detected components (cleaned) | focused, relevant, clean, best expected results |

**Methodological note:** D combines both modalities in one transformation call.
JSON provides explicit structure and component hints, while the screenshot provides
visual layout, spacing, and styling cues.

In [17]:
import os
import re
import json
import time
import base64
import urllib.request
import urllib.error
from pathlib import Path
from dotenv import load_dotenv
from collections import defaultdict

In [18]:
FIGMA_JSON_DIR  = 'dataset/figma-data/cleaned'
SCREENSHOTS_DIR = 'dataset/components'

OUTPUT_DIR      = 'dataset/storybook/src/stories'

DOCS_DIR_RAW     = 'primevue/component-documentation/raw'
DOCS_DIR_CLEANED = 'primevue/component-documentation/cleaned'

API_URL    = 'https://api.openai.com/v1/responses'
API_MODEL  = 'gpt-5.3-codex'
MAX_TOKENS = 4096
IMAGE_DETAIL = 'high'   # 'low' (85 tokens, cheaper) or 'high' (higher quality)

# Costs per token (codex-mini-latest: Input: $1.75 / Output: $14.00) (divided by one million)
INPUT_COSTS_PER_TOKEN  = 0.00000175
OUTPUT_COSTS_PER_TOKEN = 0.000014

load_dotenv(dotenv_path=Path('.env'))

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if OPENAI_API_KEY is None:
    raise ValueError('OPENAI_API_KEY not found in environment variables. Please set it in the .env file.')

## 1. Load data and documentations

### 1.1 Load Raw Component-Documentations

In [19]:
DOCS_DIR_PATH = Path(DOCS_DIR_RAW)
RAW_DOCS: dict[str, str] = {}

for md_file in sorted(DOCS_DIR_PATH.glob('*.md')):
    RAW_DOCS[md_file.stem.lower()] = md_file.read_text(encoding='utf-8', errors='ignore')

print(f'Loaded Docs: {len(RAW_DOCS)}')
for name, content in RAW_DOCS.items():
    tokens_est = len(content) // 4
    print(f'  {name:20s}  ~{tokens_est:5d} Tokens')

TOTAL_TOKENS_D2 = sum(len(c) // 4 for c in RAW_DOCS.values())
print(f'\nTotal D2-Context: ~{TOTAL_TOKENS_D2:,} Tokens')

Loaded Docs: 25
  accordion             ~ 8091 Tokens
  avatar                ~ 2737 Tokens
  badge                 ~ 1860 Tokens
  breadcrumb            ~  988 Tokens
  button                ~14672 Tokens
  card                  ~ 1426 Tokens
  checkbox              ~ 4414 Tokens
  datatable             ~20664 Tokens
  datepicker            ~11753 Tokens
  dialog                ~ 9667 Tokens
  divider               ~ 4877 Tokens
  inputnumber           ~ 7011 Tokens
  inputtext             ~ 6664 Tokens
  menu                  ~ 3240 Tokens
  password              ~ 8082 Tokens
  popover               ~ 3859 Tokens
  progressbar           ~ 1173 Tokens
  radiobutton           ~ 4283 Tokens
  select                ~11278 Tokens
  skeleton              ~ 3134 Tokens
  slider                ~ 2688 Tokens
  tabs                  ~ 7255 Tokens
  tag                   ~ 1966 Tokens
  textarea              ~ 6287 Tokens
  toggleswitch          ~ 2693 Tokens

Total D2-Context: ~150,762 Tokens

### 1.2 Load Cleaned Component-Documentations

In [20]:
DOCS_DIR_PATH = Path(DOCS_DIR_CLEANED)
CLEANED_DOCS: dict[str, str] = {}

for md_file in sorted(DOCS_DIR_PATH.glob('*.md')):
    CLEANED_DOCS[md_file.stem.lower()] = md_file.read_text(encoding='utf-8', errors='ignore')

print(f'Loaded Docs: {len(CLEANED_DOCS)}')
for name, content in CLEANED_DOCS.items():
    tokens_est = len(content) // 4
    print(f'  {name:20s}  ~{tokens_est:5d} Tokens')

TOTAL_TOKENS_D3 = sum(len(c) // 4 for c in CLEANED_DOCS.values())
print(f'\nTotal D3-Context: ~{TOTAL_TOKENS_D3:,} Tokens')

Loaded Docs: 25
  accordion             ~ 1811 Tokens
  avatar                ~  432 Tokens
  badge                 ~  283 Tokens
  breadcrumb            ~  169 Tokens
  button                ~ 3820 Tokens
  card                  ~  501 Tokens
  checkbox              ~  626 Tokens
  datatable             ~ 2480 Tokens
  datepicker            ~ 1755 Tokens
  dialog                ~ 1349 Tokens
  divider               ~  725 Tokens
  inputnumber           ~ 1634 Tokens
  inputtext             ~ 3649 Tokens
  menu                  ~  539 Tokens
  password              ~ 4501 Tokens
  popover               ~  761 Tokens
  progressbar           ~  272 Tokens
  radiobutton           ~  571 Tokens
  select                ~ 2065 Tokens
  skeleton              ~  438 Tokens
  slider                ~  487 Tokens
  tabs                  ~ 1641 Tokens
  tag                   ~  360 Tokens
  textarea              ~ 3549 Tokens
  toggleswitch          ~  587 Tokens

Total D3-Context: ~35,005 Tokens


### 1.3 Load Figma JSON Data

In [21]:
INPUT_JSON_DATA_DIR_PATH = Path(FIGMA_JSON_DIR)
FIGMA_DATA: dict[str, dict] = {}

json_data_input_files = sorted(
    f for f in INPUT_JSON_DATA_DIR_PATH.rglob('*.json')
    if f.parent != INPUT_JSON_DATA_DIR_PATH
)

# For testing: only load one file per complexity level
#seen_complexities = set()
#filtered = []
#for f in json_data_input_files:
#    complexity = f.parent.name
#
#    if complexity not in seen_complexities:
#        seen_complexities.add(complexity)
#        filtered.append(f)
#
#json_data_input_files = filtered

for input_file in json_data_input_files:
    print(f'Loading {input_file}...')

    with open(input_file, encoding='utf-8-sig') as f:
        key = f'{input_file.parent.name}-{input_file.stem}'.lower()
        FIGMA_DATA[key] = json.load(f)

print(f'\nLoaded Figma JSONs: {len(FIGMA_DATA)}')
for name, data in FIGMA_DATA.items():
    tokens_est = len(json.dumps(data)) // 4

    print(f'  {name:20s}  ~{tokens_est:5d} Tokens')

Loading dataset\figma-data\cleaned\hard\1.json...
Loading dataset\figma-data\cleaned\hard\10.json...
Loading dataset\figma-data\cleaned\hard\2.json...
Loading dataset\figma-data\cleaned\hard\3.json...
Loading dataset\figma-data\cleaned\hard\4.json...
Loading dataset\figma-data\cleaned\hard\5.json...
Loading dataset\figma-data\cleaned\hard\6.json...
Loading dataset\figma-data\cleaned\hard\7.json...
Loading dataset\figma-data\cleaned\hard\8.json...
Loading dataset\figma-data\cleaned\hard\9.json...
Loading dataset\figma-data\cleaned\medium\1.json...
Loading dataset\figma-data\cleaned\medium\10.json...
Loading dataset\figma-data\cleaned\medium\2.json...
Loading dataset\figma-data\cleaned\medium\3.json...
Loading dataset\figma-data\cleaned\medium\4.json...
Loading dataset\figma-data\cleaned\medium\5.json...
Loading dataset\figma-data\cleaned\medium\6.json...
Loading dataset\figma-data\cleaned\medium\7.json...
Loading dataset\figma-data\cleaned\medium\8.json...
Loading dataset\figma-data\cle

### 1.4 Load Screenshots

In [22]:
SCREENSHOTS_DIR_PATH = Path(SCREENSHOTS_DIR)
SCREENSHOTS: dict[str, str] = {}

screenshot_input_files = sorted(
    f for f in SCREENSHOTS_DIR_PATH.rglob('*.png')
    if f.parent != SCREENSHOTS_DIR_PATH
)

# For testing: only load one file per complexity level
#seen_complexities = set()
#filtered = []
#for f in screenshot_input_files:
#    complexity = f.parent.name
#
#    if complexity not in seen_complexities:
#        seen_complexities.add(complexity)
#        filtered.append(f)
#
#screenshot_input_files = filtered

for input_file in screenshot_input_files:
    print(f'Loading {input_file}...')

    with open(input_file, 'rb') as f:
        key = f'{input_file.parent.name}-{input_file.stem}'.lower()
        SCREENSHOTS[key] = base64.b64encode(f.read()).decode('utf-8')

print(f'\nLoaded Screenshots: {len(SCREENSHOTS)}')
for name, content in SCREENSHOTS.items():
    size_kb = len(content) / 1024
    print(f'  {name:20s}  {size_kb:6.1f} KB')

Loading dataset\components\hard\1.png...
Loading dataset\components\hard\10.png...
Loading dataset\components\hard\2.png...
Loading dataset\components\hard\3.png...
Loading dataset\components\hard\4.png...
Loading dataset\components\hard\5.png...
Loading dataset\components\hard\6.png...
Loading dataset\components\hard\7.png...
Loading dataset\components\hard\8.png...
Loading dataset\components\hard\9.png...
Loading dataset\components\medium\1.png...
Loading dataset\components\medium\10.png...
Loading dataset\components\medium\2.png...
Loading dataset\components\medium\3.png...
Loading dataset\components\medium\4.png...
Loading dataset\components\medium\5.png...
Loading dataset\components\medium\6.png...
Loading dataset\components\medium\7.png...
Loading dataset\components\medium\8.png...
Loading dataset\components\medium\9.png...
Loading dataset\components\simple\1.png...
Loading dataset\components\simple\10.png...
Loading dataset\components\simple\2.png...
Loading dataset\components\s

## 2. Detect PrimeVue-Components in Figma Data

In [23]:
KNOWN_COMPONENTS = set(CLEANED_DOCS.keys())

# Compound components that appear as FRAME
FRAME_COMPONENTS = {
    'card', 'dialog', 'tabs', 'datatable', 'select',
    'popover', 'breadcrumb', 'accordion',
}

# Aliases (Figma-Name -> Doc-Name)
DOC_ALIASES = {
    'calendar': 'datepicker',
    'overlaybadge': 'badge',
}


def _normalize(name: str) -> str:
    return re.sub(r'[\s\-_]+', '', name or '').lower()


def detect_components(figma_node: dict, found: set | None = None) -> set[str]:
    """Collect all PrimeVue component names that appear in the Figma JSON."""
    if found is None:
        found = set()

    if not isinstance(figma_node, dict):
        return found

    name = figma_node.get('name', '')
    t = figma_node.get('type', '')

    if name.startswith('_'):
        return found  # internal sub-instance

    norm = _normalize(name)
    norm = DOC_ALIASES.get(norm, norm)

    if t in ('INSTANCE', 'FRAME') and norm in KNOWN_COMPONENTS:
        found.add(norm)

    for child in figma_node.get('children', []) or []:
        detect_components(child, found)

    return found


DETECTED_COMPONENTS: dict[str, set[str]] = {}

for key, data in FIGMA_DATA.items():
    detected = detect_components(data)
    DETECTED_COMPONENTS[key] = detected

    print(f'{key:20s}  -> Detected: {", ".join(sorted(detected)) or "None"}')

print(f'\nTotal Unique Detected Components: {len(set.union(*DETECTED_COMPONENTS.values()))}')

hard-1                -> Detected: button, checkbox, dialog, inputtext, select
hard-10               -> Detected: button, datatable, popover
hard-2                -> Detected: datatable
hard-3                -> Detected: button, datepicker, inputnumber, inputtext, select
hard-4                -> Detected: button, datatable, popover
hard-5                -> Detected: button, dialog, tabs
hard-6                -> Detected: inputtext, select
hard-7                -> Detected: avatar, button, card, divider, tag
hard-8                -> Detected: button, card, tabs
hard-9                -> Detected: avatar, breadcrumb, button, datepicker, dialog, divider, inputtext, textarea
medium-1              -> Detected: button, card, checkbox, inputtext, password
medium-10             -> Detected: button, card, progressbar
medium-2              -> Detected: tabs
medium-3              -> Detected: accordion
medium-4              -> Detected: breadcrumb, button, menu
medium-5              -> Detected: b

## 3. Prompt Building

### 3.1 Context-Builder

In [24]:
def build_context(strategy: str, figma_node: dict | None = None) -> tuple[str, list[str]]:
    """Returns (context_string, used_components).

    strategy: 'd1' | 'd2' | 'd3'
    figma_node: Required for d2 and d3 for component recognition
    """

    if strategy == 'd1':
        return '', []  # No documentation context for D1

    if figma_node is None:
        raise ValueError('Strategy requires figma_node for component recognition.')

    docs = RAW_DOCS if strategy == 'd2' else CLEANED_DOCS

    detected_components = detect_components(figma_node)

    context_parts = []
    used_components = []

    for comp in sorted(detected_components):
        if comp in docs:
            print(f'  Adding doc for component: {comp} ({len(docs[comp]) // 4} tokens)')
            context_parts.append(f'# {comp}\n\n{docs[comp]}')
            used_components.append(comp)

    # Only docs of detected components in raw or cleaned form, depending on strategy.
    return '\n\n'.join(context_parts), used_components

### 3.2 Prompt Templates

In [25]:
SYSTEM_PROMPT_TEMPLATE_ZERO_SHOT = """You are an expert Vue 3 and PrimeVue developer.
Transform the given Figma mockup into a complete, working Vue 3 Single File Component
by combining two inputs:
1) Structured metadata (Figma JSON)
2) Visual evidence (screenshot)

STRICT REQUIREMENTS:
- Use PrimeVue 4 components exclusively for all UI elements visible in the screenshot and represented in JSON
- Use <script setup> syntax (no Options API)
- Import every PrimeVue component used: import Button from 'primevue/button'
- Use Tailwind CSS utility classes for layout and spacing
- Use reactive() from Vue for all form/input state
- Map Figma Auto-Layout (HORIZONTAL/VERTICAL) to flex/flex-col and map spacing/padding to gap-/p- classes
- Output ONLY the Vue SFC - no explanation, no markdown fences, no prose
- Return exactly one complete Vue SFC, starting directly with <template> and ending with </script>
- If screenshot and JSON conflict, prefer screenshot for visual appearance and JSON for structural intent and component identity
- Before finalizing, verify that the SFC is syntactically valid, all used PrimeVue components are imported, and all form/input state uses reactive()

INPUT GUIDANCE:
- Use JSON for hierarchy, component detection, and explicit properties
- Use screenshot for labels, visual grouping, spacing, and style cues
- Nodes with name starting with '_' are internal sub-instances and can be ignored

PrimeVue DOCUMENTATION:
{context}"""

USER_TEXT_TEMPLATE = """Transform this mockup into a Vue 3 SFC with PrimeVue.
Use both the screenshot and JSON below.

Figma Mockup JSON:
```json
{figma_json}
```"""

### 3.3 Prompt Builder

In [26]:
def _remove_outer_figma_frame(figma_root: dict) -> dict:
    """If root is a FRAME with one child, unwrap it to reduce nesting noise."""
    if figma_root.get('type') == 'FRAME' and len(figma_root.get('children', [])) == 1:
        return figma_root['children'][0]

    return figma_root


def build_prompts(figma_root: dict, strategy: str) -> tuple[str, str, list[str], int]:
    """Creates system and user text prompts.

    Returns: (system_prompt, user_text, used_components, context_tokens)
    """
    figma_root = _remove_outer_figma_frame(figma_root)

    context, used_components = build_context(strategy, figma_root)
    context_tokens = len(context) // 4

    system_prompt = SYSTEM_PROMPT_TEMPLATE_ZERO_SHOT.format(
        context=context if context else '(No documentation provided use pretrained knowledge)'
    )

    user_text = USER_TEXT_TEMPLATE.format(
        figma_json=json.dumps(figma_root, ensure_ascii=False, indent=2)
    )

    return system_prompt, user_text, used_components, context_tokens

## 4. LLM Interaction (Hybrid)

In [27]:
def call_llm_hybrid(system_prompt: str, user_text: str, base64_image: str,
                    strategy: str, key: str) -> dict:
    """Calls the OpenAI Responses API with combined image + text input.

    Returns: {
        'content':      str,
        'input_tokens': int,
        'output_tokens':int,
        'stop_reason':  str,
        'duration':     float,
    }
    """
    metadata = {
        'strategy':   strategy,
        'mockup_key': key,
    }

    payload = json.dumps({
        'model': API_MODEL,
        'metadata': metadata,
        'input': [
            {'role': 'system', 'content': system_prompt},
            {
                'role': 'user',
                'content': [
                    {
                        'type': 'input_image',
                        'image_url': f'data:image/png;base64,{base64_image}',
                        'detail': IMAGE_DETAIL,
                    },
                    {
                        'type': 'input_text',
                        'text': user_text,
                    },
                ],
            },
        ],
    }).encode('utf-8')

    headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {OPENAI_API_KEY}',
    }

    req = urllib.request.Request(
        API_URL,
        data=payload,
        headers=headers,
        method='POST',
    )

    start_time = time.time()

    try:
        with urllib.request.urlopen(req) as resp:
            resp_data = json.loads(resp.read().decode('utf-8'))
    except urllib.error.HTTPError as e:
        error_text = e.read().decode('utf-8', errors='ignore')

        try:
            error_body = json.loads(error_text)
            detail = error_body.get('error', {}).get('message', error_text)
        except json.JSONDecodeError:
            detail = error_text or str(e)

        raise RuntimeError(f'OpenAI API Fehler {e.code}: {detail}') from e

    end_time = time.time()

    usage = resp_data.get('usage', {})

    content = resp_data['output'][0]['content'][0]['text']
    input_tokens = usage.get('input_tokens', 0)
    output_tokens = usage.get('output_tokens', 0)
    stop_reason = resp_data['output'][0].get('status', '')

    return {
        'content': content,
        'input_tokens': input_tokens,
        'output_tokens': output_tokens,
        'stop_reason': stop_reason,
        'duration': end_time - start_time,
    }

In [28]:
def extract_sfc(raw: str) -> str:
    """Extracts the Vue SFC code from the LLM response

    Handles three cases:

    1. Direct SFC output (starts with <template>)
    2. Code block with language annotation (```vue ... ```)
    3. Generic code block (``` ... ```)
    """
    # Case 2: ```vue ... ```
    m = re.search(r'```vue\s*\n(.+?)```', raw, re.DOTALL)
    if m:
        return m.group(1).strip()

    # Case 3: ``` ... ```
    m = re.search(r'```\s*\n(.+?)```', raw, re.DOTALL)
    if m:
        candidate = m.group(1).strip()

        if '<template>' in candidate:
            return candidate

    # Case 1: Direct SFC
    if '<template>' in raw:
        start = raw.index('<template>')

        return raw[start:].strip()

    # Fallback: Return the raw response with a comment
    return f'<!-- SFC-Extraktion failed -->\n<!-- RAW:\n{raw[:500]}\n-->'

## 5. Record metrics

In [29]:
_metrics_d: dict = {}

def _reset_metrics_d():
    global _metrics_d
    _metrics_d = {
        'input_tokens': 0,
        'output_tokens': 0,
        'context_tokens': 0,
        'context_components': 0,
        'stop_reason': '',
        'duration': 0,
        'parse_ok': False,
    }


def _ast_depth_approx(sfc: str) -> int:
    """Estimates max template depth by counting opening/closing tags."""
    depth, max_depth = 0, 0
    in_template = False

    for line in sfc.splitlines():
        if '<template>' in line:
            in_template = True

        if not in_template:
            continue

        depth += line.count('<') - line.count('</') - line.count('/>')
        max_depth = max(max_depth, depth)

    return max(0, max_depth)

## 6. Main transformation for one mockup and one strategy

In [30]:
def generate_sfc_d(figma_root: dict, screenshot: str, strategy: str, key: str) -> str:
    """Transforms one mockup with the specified hybrid strategy.

    strategy: 'd1' | 'd2' | 'd3'
    Returns: Vue 3 SFC as string
    """
    _reset_metrics_d()

    system_prompt, user_text, used_components, context_tokens = build_prompts(figma_root, strategy)

    print(f'Detected Components: {used_components} -> Context Tokens: {context_tokens}')

    _metrics_d['context_tokens'] = context_tokens
    _metrics_d['context_components'] = len(used_components)

    response = call_llm_hybrid(system_prompt, user_text, screenshot, strategy, key)

    _metrics_d['input_tokens'] = response['input_tokens']
    _metrics_d['output_tokens'] = response['output_tokens']
    _metrics_d['stop_reason'] = response['stop_reason']
    _metrics_d['duration'] = response['duration']

    sfc = extract_sfc(response['content'])

    _metrics_d['parse_ok'] = '<template>' in sfc and '<script' in sfc

    return sfc

## 7. Pipeline for all mockups and all strategies

In [31]:
STRATEGIES = ['d1', 'd2', 'd3']
OUTPUT_PATH = Path(OUTPUT_DIR)

all_results: list[dict] = []

print(f'Input-Files: {len(FIGMA_DATA)} Strategies: {STRATEGIES}\n')


def _complexity_from_key(key: str) -> str:
    """Extract complexity prefix from keys like 'medium-8' or 'simple-5'."""
    return key.split('-', 1)[0] if '-' in key else 'unknown'


for key, figma_root in FIGMA_DATA.items():
    complexity = _complexity_from_key(key)

    print(f' Processing {key}...\n{"=" * 60}')

    for strategy in STRATEGIES:

        try:
            print(f' Strategy: {strategy.upper()}')

            sfc = generate_sfc_d(figma_root, SCREENSHOTS[key], strategy, key)

            out_path = OUTPUT_PATH / complexity / f'{key.split("-", 1)[1]}-{strategy}.vue'
            out_path.parent.mkdir(parents=True, exist_ok=True)
            out_path.write_text(sfc, encoding='utf-8')

            m = dict(_metrics_d)
            result = {
                'input': key,
                'output': out_path.name,
                'complexity': complexity,
                'strategy': strategy,
                'image_detail': IMAGE_DETAIL,
                'duration_ms': round(m['duration'] * 1000, 4),
                'sfc_bytes': len(sfc),
                'sfc_lines': sfc.count('\n') + 1,
                'ast_depth_approx': _ast_depth_approx(sfc),
                'parse_ok': m['parse_ok'],
                'input_tokens': m['input_tokens'],
                'output_tokens': m['output_tokens'],
                'context_tokens': m['context_tokens'],
                'context_components': m['context_components'],
                'stop_reason': m['stop_reason'],
                'cost_usd': round(
                    m['input_tokens'] * INPUT_COSTS_PER_TOKEN +
                    m['output_tokens'] * OUTPUT_COSTS_PER_TOKEN,
                    6,
                ),
                'error': None,
            }

            print(
                f'  OK  {key:25s} [{strategy}]  '
                f'in={m["input_tokens"]:5d}tok  '
                f'out={m["output_tokens"]:4d}tok  '
                f'${result["cost_usd"]:.4f}  '
                f'{m["duration"]:6.0f}ms'
            )

        except Exception as e:
            print(f'  ERROR {key:25s} [{strategy}]  {str(e)}')

            result = {
                'input': key,
                'output': None,
                'complexity': complexity,
                'strategy': strategy,
                'image_detail': IMAGE_DETAIL,
                'error': str(e),
                **{k: None for k in [
                    'duration_ms', 'sfc_bytes', 'sfc_lines', 'ast_depth_approx',
                    'parse_ok', 'input_tokens', 'output_tokens', 'context_tokens',
                    'context_components', 'stop_reason', 'cost_usd'
                ]},
            }

        all_results.append(result)


print(f'\nCompleted {len(all_results)} transformed')

Input-Files: 30 Strategies: ['d1', 'd2', 'd3']

 Processing hard-1...
 Strategy: D1
Detected Components: [] -> Context Tokens: 0
  OK  hard-1                    [d1]  in= 8083tok  out= 575tok  $0.0222       7ms
 Strategy: D2
  Adding doc for component: button (14672 tokens)
  Adding doc for component: checkbox (4414 tokens)
  Adding doc for component: dialog (9667 tokens)
  Adding doc for component: inputtext (6664 tokens)
  Adding doc for component: select (11278 tokens)
Detected Components: ['button', 'checkbox', 'dialog', 'inputtext', 'select'] -> Context Tokens: 46713
  OK  hard-1                    [d2]  in=54592tok  out= 807tok  $0.1068      11ms
 Strategy: D3
  Adding doc for component: button (3820 tokens)
  Adding doc for component: checkbox (626 tokens)
  Adding doc for component: dialog (1349 tokens)
  Adding doc for component: inputtext (3649 tokens)
  Adding doc for component: select (2065 tokens)
Detected Components: ['button', 'checkbox', 'dialog', 'inputtext', 'select']

## 8. Save report

In [32]:
def _avg(vals):
    clean = [v for v in vals if v is not None]

    return round(sum(clean) / len(clean), 4) if clean else None


# Aggregation: per Strategy
by_strategy = defaultdict(list)
for r in all_results:
    by_strategy[r['strategy']].append(r)

per_strategy = {}
for s, items in by_strategy.items():
    ok_items = [i for i in items if not i['error']]
    per_strategy[s] = {
        'count': len(items),
        'errors': len(items) - len(ok_items),
        'parse_ok_rate': _avg([i['parse_ok'] for i in ok_items]),
        'avg_duration_ms': _avg([i['duration_ms'] for i in ok_items]),
        'avg_input_tokens': _avg([i['input_tokens'] for i in ok_items]),
        'avg_output_tokens': _avg([i['output_tokens'] for i in ok_items]),
        'avg_context_tokens': _avg([i['context_tokens'] for i in ok_items]),
        'total_cost_usd': round(sum(i['cost_usd'] or 0 for i in ok_items), 4),
        'avg_cost_usd_per_file': _avg([i['cost_usd'] for i in ok_items]),
    }

# Aggregation: per Strategy x Complexity
by_strat_complexity = defaultdict(lambda: defaultdict(list))
for r in all_results:
    if not r['error']:
        by_strat_complexity[r['strategy']][r['complexity']].append(r)

per_strategy_complexity = {}
for s, levels in by_strat_complexity.items():
    per_strategy_complexity[s] = {
        lvl: {
            'count': len(items),
            'avg_cost_usd': _avg([i['cost_usd'] for i in items]),
            'avg_input_tokens': _avg([i['input_tokens'] for i in items]),
            'avg_output_tokens': _avg([i['output_tokens'] for i in items]),
            'avg_duration_ms': _avg([i['duration_ms'] for i in items]),
            'parse_ok_rate': _avg([i['parse_ok'] for i in items]),
        }
        for lvl, items in levels.items()
    }

metrics_report = {
    'method': 'D',
    'model': API_MODEL,
    'image_detail': IMAGE_DETAIL,
    'prompt_strategy': 'zero_shot',
    'strategies_run': STRATEGIES,
    'per_strategy': per_strategy,
    'per_strategy_complexity': per_strategy_complexity,
    'files': all_results,
}

report_path = Path('reports') / 'metrics_report_d.json'
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(metrics_report, f, indent=4, ensure_ascii=False)

print(f'Transformation report saved to: {report_path}')


Transformation report saved to: reports\metrics_report_d.json
